In [ ]:
"""
Load and preprocess the files to a protobuff
"""

import numpy as np
from distutils.util import strtobool
from random import shuffle
import matplotlib
import nibabel as nb
from pathlib import Path
import cv2
import pickle
#import matplotlib.pyplot as plt
import glob
import h5py
import zipfile
import io
import os.path

home_dir = ''


def saveCompressed(fh, **namedict):
    with zipfile.ZipFile(fh,
                         mode="w",
                         compression=zipfile.ZIP_DEFLATED,
                         allowZip64=True) as zf:
        for k, v in namedict.items():
            buf = io.BytesIO()
            np.lib.npyio.format.write_array(buf,
                                            np.asanyarray(v),
                                            allow_pickle=True)
            zf.writestr(k + '.npy',
                        buf.getvalue())


def window_image(volume, level, width):
    """
    Windows the image. For images that come from CT scans
    :param volume: ndarray, Source image
    :param level: the center of the window
    :param width: how wide the window is +-
    :return: the windowed image
    """

    # Set the min and max HU
    minimum = int(level - width)
    maximum = int(level + width)

    # Fit to max
    volume[volume > maximum] = maximum

    # fit to min
    volume[volume < minimum] = minimum

    # Return the windowed image
    return volume

def threshold(volume, maximum):
    maximum = maximum
    #volume[volume < minimum] = 0
    volume[volume > maximum] = 1
    return volume

def retreive_filelist(extension, path, include_subfolders=True):
    """
    Returns a list with all the files of a certain type in path
    :param extension: what extension to search for
    :param include_subfolders: whether to include subfolders
    :param path: specified path. otherwise use data root
    :return:
    """
    if include_subfolders:
        extension = ('**/**/*.%s' % extension)

    # Otherwise just search this folder
    else:
        extension = ('*.%s' % extension)

    # Join the pathnames
    path = os.path.join(path, extension)

    # Return the list of filenames
    return glob.glob(path)

def reshape_NHWC(vol, NHWC):
    """
    Method to reshape 2D or 3D tensor into Tensorflow's NHWC format
    vol: The image data
    NHWC whether the input has a channel dimension
    """

    # If this is a 2D image
    if len(vol.shape) == 2:
        # Create an extra channel at the beginning
        vol = np.expand_dims(vol, axis=0)

    # If there are 3 dimensions to the shape (2D with channels or 3D)
    if len(vol.shape) == 3:

        # If there is no channel dimension (i.e. grayscale)
        if not NHWC:
            # Move the last axis (Z) to the first axis
            vol = np.moveaxis(vol, -1, 0)

        # Create another axis at the end for channel
        vol = np.expand_dims(vol, axis=3)

    return vol

def load_NIFTY(path, reshape=True):
    """
    This function loads a .nii.gz file into a numpy array with dimensions Z, Y, X, C
    :param filename: path to the file
    :param reshape: whether to reshape the axis from/to ZYX
    :return:
    """

    # try:

    # Load the data from the nifty file
    raw_data = nb.load(path)

    # Reshape the image data from NiB's XYZ to numpy's ZYXC
    if reshape:
        data = reshape_NHWC(raw_data.get_data(), False)
    else:
        data = raw_data.get_data()

    # Return the data
    return data


def zoom_2d(image, new_shape):
    """
    Uses open CV to resize a 2D image
    :param image: The input image, numpy array
    :param new_shape: New shape, tuple or array
    :return: the resized image
    """
    return cv2.resize(image, (new_shape[0], new_shape[1]), interpolation=cv2.INTER_NEAREST)


def pre_proc_25D(slice_gap, dims):
    filenames1, filenames = retreive_filelist('nii.gz', home_dir, include_subfolders=False), []
    for z in range(len(filenames1)):
        filenames.append(filenames1[z])
    print(len(filenames), ' files found. Examples: ', filenames)

    # global variables
    index, pts, per = 0, 0, 0
    index1, pts1, per1 = 0, 0, 0
    data = {}
    data1 = {}
    patients = []
    patient1 =[]
    studies = []

    for file in filenames:
        if not file.endswith("_Label_1.nii.gz"):
            patient2 = file.split('/')[4]
        # First retreive the patient name

        #print(patient2)
            patients.append(patient2)


        for patient2 in patients:
            if patient2 not in patient1:
                patient1.append(patient2)
            study = file.split('/')[4]
            studies.append(study)

        train = patient1[0:440]
        val = patient1[440:550]

    print(len(patient1))
    for file in filenames:
        if not file.endswith("_Label_1.nii.gz"):
            patient = file.split('/')[4]
            if patient in val:
                study = file.split('/')[4]
                bleed_segments_file = file.split('.')[0] + '_Label_1.nii.gz'
            # Now load the volumes
                try:
                    volume = np.squeeze(load_NIFTY(file))
                    bleed_segment = np.squeeze(load_NIFTY(bleed_segments_file))
                    print('loading:', file)
                except:
                    print('Unable to load: ', file)
                    continue

                volume = window_image(volume, 80, 80)
                edema_segment = threshold(edema_segment, 1)
                print(np.min(volume))
                print(np.max(volume))

                segments = bleed_segment

                for z in range(volume.shape[0]):

                # Skip very bottom and very top of image
                    if ((z<85) or (z>184)): continue
                    data_image = zoom_2d(volume[z].astype(np.float32),
                                             [dims, dims])
                    data_label = zoom_2d(segments[z].astype(np.float32),
                                              [dims, dims])

                    num_egs = 1

                    for _ in range(num_egs):
                    # Save the dictionary: int16, int16, int, int
                        data[index] = {'image_data': data_image, 'label_data': data_label, 'patient': patient, 'study': study}

                    # Finished with this slice
                        index += 1

                # Garbage collection
                    del data_label, data_image


                pts += 1
                if pts % 30 == 0:
                # Counters
                    per = index - per

    saveCompressed(open('', 'wb'), data=data)

    print('%s Total patients loaded, %s total slices. Final data: %s slices' % (pts, index, len(data)))
    del data

    for file in filenames:
        if not file.endswith("_Label_1.nii.gz"):
            patient2 = file.split('/')[4]
            if patient2 in train:
                study1 = file.split('/')[4]
                bleed_segments_file1 = file.split('.')[0] + '_Label_1.nii.gz'
            # Now load the volumes
                try:
                    volume1 = np.squeeze(load_NIFTY(file))
                    bleed_segment1 = np.squeeze(load_NIFTY(bleed_segments_file1))
                    print('loading:', file)
                except:
                    print('Unable to load: ', file)
                    continue

            #Window the volume and center it at 0
                volume1 = window_image(volume1, 80, 80)
                bleed_segment1 = threshold(bleed_segment1, 1)
                print(np.min(volume1))
                print(np.max(volume1))
                segments1 = bleed_segment1

                for a in range(volume1.shape[0]):

                # Skip very bottom and very top of image
                    if ((a<85) or (a>185)): continue
                    data_image1 = np.zeros(shape=[1, dims, dims], dtype=np.float32)
                    data_label1 = np.zeros(shape=[1, dims, dims], dtype=np.float32)
                    data_image1 = zoom_2d(volume1[a].astype(np.float32),
                                              [dims, dims])
                    data_label1 = zoom_2d(segments1[a].astype(np.float32),
                                              [dims, dims])
                    num_egs1 = 1

                    for _ in range(num_egs1):
                # Save the dictionary: int16, int16, int, int
                        data1[index1] = {'image_data': data_image1, 'label_data': data_label1, 'patient': patient1, 'study': study1}
                    #print(patient1)
                    # Finished with this slice
                        index1 += 1

                # Garbage collection
                    del data_label1, data_image1

            pts1 += 1
            if pts1 % 30 == 0:
                # Counters
                per1 = index1 - per1

    saveCompressed(open('', 'wb'), data=data1)

    del data1
    print('%s Total patients loaded, %s total slices. Final data: %s slices' % (pts, index, len(data1)))

In [ ]:
from __future__ import division, print_function
from collections import defaultdict
import os, pickle, sys
from scipy import misc

import shutil
from functools import partial
import numpy as np
from PIL import Image
from random import shuffle
from skimage.exposure import equalize_adapthist, equalize_hist
from skimage.transform import resize
import imageio
from scipy import misc
import zipfile
import io
import os.path


studies = []
masks_train=[]
images_train=[]
masks_train1=[]
images_train1=[]
patients1=[]
patients2=[]
slices =[]

with np.load('', allow_pickle=True) as data:
        a = data['data']
        totalfiles = len(a.all())
        for i in range(totalfiles):
            im1 = a.all()[i]['image_data'];
            mask = a.all()[i]['label_data'];
            mask.astype(np.uint8))
            im1 = np.squeeze(im1[:, :]);
            mask = np.squeeze(mask[:, :]);
            images_train.append(im1)
            masks_train.append(mask)

np.save('', masks_train)
np.save('', images_train)

with np.load('', allow_pickle=True) as data:
        a = data['data']
        totalfiles = len(a.all())
        for i in range(totalfiles):
            im2 = a.all()[i]['image_data'];
            mask1 = a.all()[i]['label_data'];
            mask1.astype(np.uint8))
            im2 = np.squeeze(im2[:, :]);
            mask1 = np.squeeze(mask1[:, :]);
            images_train1.append(im1)
            masks_train1.append(mask)

np.save('', masks_train1)
np.save('', images_train1)